# L02 – Tensorer och autograd

**Tid:** ca 2 h · **Del A: Grunder**

## Mål
1. Skapa tensorer och förstå **shape**, **dtype** och **device**.
2. Använda broadcasting och flytta data mellan NumPy och PyTorch.
3. Förstå vad en **gradient** är och hur `backward()` beräknar den.
4. Veta varför `zero_grad()` och `torch.no_grad()` behövs.

**Var i arbetsflödet?** Det här är verktygen som allt annat bygger på. Varje steg i träningen (L03) är tensoroperationer med autograd.

In [ ]:
import random
import numpy as np
import torch

def set_seed(seed: int = 42) -> None:
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## 1. Vad är en tensor?

En **tensor** är en flerdimensionell array, nästan som en NumPy-array, men med två tillägg:
1. den kan ligga på **GPU**,
2. den kan hålla reda på hur den beräknades, så att PyTorch kan räkna ut **gradienter** automatiskt (autograd).

| Dimensioner | Exempel | Shape |
|---|---|---|
| 0 (skalär) | en loss | `()` |
| 1 (vektor) | etiketter för en batch | `(N,)` |
| 2 (matris) | tabelldata, som `X` i sklearn | `(N, features)` |
| 4 | en batch bilder | `(N, kanaler, höjd, bredd)` |

In [ ]:
a = torch.tensor([1.0, 2.0, 3.0])         # från en lista
b = torch.zeros(2, 3)                     # nollor
c = torch.rand(2, 3)                      # likformigt slump i [0, 1)
d = torch.arange(6)                       # 0, 1, ..., 5
images = torch.rand(32, 1, 28, 28)        # 32 gråskalebilder, 28x28 pixlar

for name, t in [("a", a), ("b", b), ("c", c), ("d", d), ("images", images)]:
    print(f"{name:7s} shape={tuple(t.shape)!s:16s} dtype={t.dtype}  device={t.device}")

### dtype – varför `float32`?
- **`float32`** är standard för vikter och indata. Precisionen räcker för neurala nät, och GPU:er är optimerade för det. `float64` (som NumPy använder som standard) är dubbelt så stort och mycket långsammare på GPU.
- **`int64`** (`torch.long`) används för **klassetiketter** och index. `CrossEntropyLoss` (L03) kräver det.

En vanlig nybörjarbugg är att blanda dtypes:

In [ ]:
x_np = np.array([1.0, 2.0])          # NumPy ger float64
x = torch.from_numpy(x_np)
print("från NumPy:", x.dtype)        # float64!
x = x.float()                        # konvertera till float32
print("efter .float():", x.dtype)

labels = torch.tensor([0, 2, 1])
print("etiketter:", labels.dtype)    # int64, rätt för klassificering

## 2. Shape-operationer

Den vanligaste källan till fel i djupinlärning är **fel shape**. De här operationerna används hela tiden:

| Operation | Vad den gör | Exempel på användning |
|---|---|---|
| `reshape` / `view` | ny shape, samma data | platta ut en bild till en vektor inför ett `Linear`-lager |
| `unsqueeze(dim)` | lägg till en dimension av storlek 1 | `(N,)` blir `(N, 1)` för regressionsmål |
| `squeeze(dim)` | ta bort en dimension av storlek 1 | `(N, 1)` blir `(N,)` |
| `@` | matrismultiplikation | `X @ W` i ett lager |

In [ ]:
flat = images.reshape(32, -1)   # -1 betyder "räkna ut själv": 1*28*28 = 784
print("utplattad:", flat.shape)

y = torch.rand(32)
print("y:", y.shape, "→ unsqueeze(1):", y.unsqueeze(1).shape)

X = torch.rand(32, 784)   # 32 exempel, 784 features
W = torch.rand(784, 10)   # vikter: 784 in, 10 ut
print("X @ W:", (X @ W).shape)  # (32, 10): 10 tal per exempel

**Batchdimensionen först.** I PyTorch är första dimensionen nästan alltid *antal exempel i batchen* (`N`), precis som rader i `X` i sklearn. Ett lager behandlar alla exempel i batchen parallellt.

## 3. Broadcasting

När två tensorer har olika shape "sträcker" PyTorch ut den mindre så att de passar ihop, enligt samma regler som i NumPy. Det används till exempel för att standardisera kolumner. Det är exakt vad `StandardScaler` gör:

In [ ]:
data = torch.tensor([[1.0, 100.0],
                     [2.0, 200.0],
                     [3.0, 300.0]])
mean = data.mean(dim=0)          # medel per kolumn, shape (2,)
std = data.std(dim=0)            # std per kolumn, shape (2,)
print("mean:", mean, " std:", std)

standardized = (data - mean) / std   # (3, 2) - (2,): mean dras av från varje rad
print(standardized)

`dim=0` betyder "längs rad-axeln", det vill säga beräkna över alla rader och få ett värde per kolumn. Det motsvarar `axis=0` i NumPy.

## 4. NumPy-bryggan och device

In [ ]:
t = torch.arange(3)
n = t.numpy()             # tensor → NumPy (delar minne på CPU!)
t[0] = 100
print("NumPy-arrayen ändrades också:", n)

t_gpu = torch.rand(3, device=device)
# t_gpu.numpy() ger fel på GPU: NumPy kan bara läsa CPU-minne
print("tillbaka till NumPy:", t_gpu.cpu().numpy())

loss = torch.tensor(0.4321)
print("skalär → Python-tal:", loss.item())  # används för att logga loss

Data och modell måste ligga på **samma device**:

In [ ]:
if device == "cuda":
    a_cpu = torch.rand(3)
    b_gpu = torch.rand(3, device="cuda")
    try:
        a_cpu + b_gpu
    except RuntimeError as e:
        print("Fel:", str(e)[:120], "...")
    print("Fungerar:", (a_cpu.to(device) + b_gpu))
else:
    print("Ingen GPU: testet hoppas över.")

## 5. Gradienter och autograd

### Vad är en gradient?
För en funktion $f(w)$ säger derivatan $f'(w)$ **hur mycket $f$ ändras om vi ökar $w$ lite**. Med många parametrar kallas vektorn av alla partiella derivator för **gradienten**. Den pekar i riktningen där $f$ ökar snabbast.

Träning går ut på att **minimera loss**. Därför tar vi ett litet steg i *motsatt* riktning mot gradienten. Det är **gradient descent**:

$$w \leftarrow w - \text{lr} \cdot \frac{\partial \text{loss}}{\partial w}$$

`lr` (learning rate) bestämmer steglängden. Ett neuralt nät kan ha miljontals parametrar, och ingen vill derivera för hand. Det sköter **autograd**.

In [ ]:
w = torch.tensor(2.0, requires_grad=True)  # "håll reda på beräkningar med w"
f = w**2 + 3*w                             # f(w) = w² + 3w
f.backward()                               # beräkna df/dw
print("df/dw vid w=2:", w.grad)            # analytiskt: 2w + 3 = 7

Vad hände?
1. `requires_grad=True` gör att PyTorch bygger en **beräkningsgraf**: $w \to w^2 \to w^2+3w$.
2. `backward()` går baklänges genom grafen och använder kedjeregeln.
3. Resultatet hamnar i `w.grad`.

### Gradient descent för hand
Vi minimerar $f(w) = (w-3)^2$, som har sitt minimum vid $w=3$:

In [ ]:
w = torch.tensor(0.0, requires_grad=True)
lr = 0.1

for step in range(25):
    loss = (w - 3) ** 2
    loss.backward()                 # beräkna gradienten
    with torch.no_grad():           # själva uppdateringen ska inte spåras
        w -= lr * w.grad
    w.grad.zero_()                  # nollställ inför nästa steg (se nedan)
    if step % 5 == 0:
        print(f"steg {step:2d}: w = {w.item():.4f}, loss = {loss.item():.4f}")
print(f"slut: w = {w.item():.4f}")

### Varför `zero_grad`? Gradienter ackumuleras
PyTorch **adderar** nya gradienter till `.grad` i stället för att skriva över dem. Det är användbart i vissa avancerade fall, men i en vanlig träningsloop måste vi nollställa varje steg:

In [ ]:
w = torch.tensor(2.0, requires_grad=True)
for i in range(3):
    f = w**2 + 3*w
    f.backward()
    print(f"efter backward nr {i+1}: w.grad = {w.grad.item()}")  # 7, 14, 21 ...

Om du glömmer att nollställa blir stegen allt större och träningen beter sig konstigt. I L03 sköter `optimizer.zero_grad()` det här.

### Varför `torch.no_grad()`?
- Vid **uppdatering av vikter**: själva uppdateringen ska inte bli en del av beräkningsgrafen.
- Vid **utvärdering och prediktion**: vi behöver inga gradienter, och det sparar minne och tid.

In [ ]:
w = torch.tensor(2.0, requires_grad=True)
print("med spårning:", (w * 3).requires_grad)
with torch.no_grad():
    print("inuti no_grad:", (w * 3).requires_grad)

## 6. Vad händer om … learning rate ändras?

**Gissa först:** vad händer med gradient descent på $(w-3)^2$ om `lr` är 0.01, 0.1, 0.9 och 1.1?

In [ ]:
def gd(lr, steps=20):
    w = torch.tensor(0.0, requires_grad=True)
    for _ in range(steps):
        loss = (w - 3) ** 2
        loss.backward()
        with torch.no_grad():
            w -= lr * w.grad
        w.grad.zero_()
    return w.item()

for lr in [0.01, 0.1, 0.9, 1.1]:
    print(f"lr={lr:<5} → w efter 20 steg: {gd(lr):10.4f}   (mål: 3)")

- **0.01**: för små steg, och vi hinner inte fram.
- **0.1**: lagom.
- **0.9**: vi studsar fram och tillbaka över minimum men konvergerar till slut.
- **1.1**: stegen blir större för varje gång och lösningen **divergerar**.

Samma fenomen ser du när du tränar riktiga nätverk. Om loss exploderar eller blir `nan` är learning rate för hög.

## 7. Övningar

### Övning 1 – shapes
En batch med 16 färgbilder (3 kanaler) på 32×32 pixlar ska plattas ut till en matris med en rad per bild.

In [ ]:
batch = torch.rand(16, 3, 32, 32)
flat_batch = None  # TODO

In [ ]:
assert flat_batch.shape == (16, 3072), f"fel shape: {flat_batch.shape}"
print("Rätt!")

### Övning 2 – standardisera med broadcasting
Standardisera kolumnerna i `M` så att varje kolumn får medel 0 och standardavvikelse 1.

In [ ]:
set_seed(1)
M = torch.rand(100, 4) * torch.tensor([1.0, 10.0, 100.0, 1000.0])
M_std = None  # TODO

In [ ]:
assert torch.allclose(M_std.mean(dim=0), torch.zeros(4), atol=1e-5)
assert torch.allclose(M_std.std(dim=0), torch.ones(4), atol=1e-5)
print("Rätt!")

### Övning 3 – gradient med autograd
Beräkna derivatan av $g(w) = w^3 - 2w$ i punkten $w = 2$ med autograd. Kontrollera för hand: $g'(w) = 3w^2 - 2$.

In [ ]:
grad_value = None  # TODO: ett Python-tal (använd .item())

In [ ]:
assert abs(grad_value - 10.0) < 1e-6
print("Rätt!")

### Övning 4 – gradient descent med två parametrar
Hitta $a$ och $b$ som minimerar $(a - 5)^2 + (b + 2)^2$ med gradient descent. Utgå från $a = b = 0$. Välj själv learning rate och antal steg.

In [ ]:
a = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
# TODO: träningsloop

In [ ]:
assert abs(a.item() - 5) < 1e-2 and abs(b.item() + 2) < 1e-2, (a.item(), b.item())
print("Rätt!")

## När använder jag detta?
- **`float32`** för indata och vikter, **`int64`** för klassetiketter.
- **`reshape` och `unsqueeze`** när ett lager eller en loss klagar på shape. Skriv ut `.shape` ofta.
- **`.to(device)`** på både modell och varje batch.
- **`.item()`** när du loggar en loss, och **`.cpu().numpy()`** när du ska använda sklearn eller matplotlib.
- **`torch.no_grad()`** vid utvärdering och prediktion.

## Sammanfattning
- En tensor har shape, dtype och device.
- Autograd bygger en beräkningsgraf, och `backward()` ger gradienten i `.grad`.
- Gradient descent: $w \leftarrow w - \text{lr} \cdot \nabla w$. Gradienter ackumuleras, så de måste nollställas varje steg.
- Learning rate är den viktigaste parametern: för liten är långsamt, för stor divergerar.

**Nästa lektion:** vi bygger en riktig träningsloop med loss-funktioner och optimerare.